## With manual log-in but automatic Chrome opening

In [ ]:
# ================================================================
# manual_login_and_extract_linkedin.py
# Run: python manual_login_and_extract_linkedin.py
# Requirements: pip install undetected-chromedriver selenium
# ================================================================

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, csv, json, os, sys
from datetime import datetime

# ---------- CONFIG ----------
profile_url = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"  # profile to scrape
driver_path = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"
wait_timeout = 300  # seconds to wait for manual login

# ---------- START BROWSER & MANUAL LOGIN ----------
options = uc.ChromeOptions()
options.add_argument("--start-maximized")  # open browser maximized
# User logs in manually; no user-data-dir used (session is temporary)

driver = uc.Chrome(driver_executable_path=driver_path, options=options)
driver.get("https://www.linkedin.com/login")
print("Please login manually in the opened browser window. The script will continue automatically once you're logged in.")

# Wait until LinkedIn session is detected
start_time = time.time()
while True:
    time.sleep(1)
    cookies = driver.get_cookies()
    if any(c.get("name") == "li_at" for c in cookies):
        print("✅ Detected LinkedIn auth cookie 'li_at' — login successful.")
        break
    if time.time() - start_time > wait_timeout:
        print(f"⚠️ No auth cookie detected after {wait_timeout} seconds.")
        ans = input("If you are already logged in, press Enter to continue anyway or type 'exit' to abort: ").strip().lower()
        if ans == "exit":
            driver.quit()
            sys.exit("Aborted by user.")
        else:
            break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(profile_url)
wait = WebDriverWait(driver, 15)

# ---------- UTILITIES ----------
def safe_click(xpath):
    """Safely click an element if it exists."""
    try:
        el = driver.find_element(By.XPATH, xpath)
        el.click()
        time.sleep(0.5)
        return True
    except Exception:
        return False

def clean_list(lst):
    """Remove duplicates and empty strings while preserving order."""
    seen = set()
    cleaned = []
    for item in lst:
        if item and item.strip() and item not in seen:
            cleaned.append(item.strip())
            seen.add(item)
    return cleaned

def safe_preview(t, n=300):
    """Truncate long text for printing."""
    return (t[:n] + "...") if t and len(t) > n else t

def try_selectors(selectors):
    """Try multiple selectors until one returns non-empty text."""
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except Exception:
            continue
    return None

def get_section_text_by_header(header_texts):
    """Find section text by matching possible header titles."""
    for label in header_texts:
        try:
            xpath = f"//h2[contains(translate(normalize-space(.), '{label.upper()}', '{label.lower()}'), '{label.lower()}')]"
            header = driver.find_element(By.XPATH, xpath)
            section = header.find_element(By.XPATH, "./ancestor::section")
            text = section.text.strip()
            if "\n" in text:
                text = text.split("\n", 1)[1].strip()
            return text
        except Exception:
            continue
    return None

# ---------- EXPAND CONTENT ----------
expand_xpaths = [
    "//button[contains(., 'See more')]",
    "//button[contains(., 'Mostra altro')]",
    "//button[contains(., 'Mostra di più')]",
    "//button[contains(., 'Vedi altro')]",
    "//button[contains(@aria-label, 'See more')]"
]
for xp in expand_xpaths:
    safe_click(xp)

# Scroll to load dynamic content
for frac in [0.25, 0.5, 0.75, 1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {frac});")
    time.sleep(0.7)

# ---------- EXTRACT FIELDS ----------
name = try_selectors([
    (By.CSS_SELECTOR, "h1.text-heading-xlarge"),
    (By.XPATH, "//main//h1")
])

headline = try_selectors([
    (By.CSS_SELECTOR, "div.text-body-medium.break-words"),
    (By.XPATH, "//main//div[contains(@class, 'text-body-medium')]")
])

location = try_selectors([
    (By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words"),
    (By.XPATH, "//main//span[contains(., ',') or contains(., 'Based')]")
])

about = get_section_text_by_header(["About", "Informazioni", "Profilo"])

# ---------- EXPERIENCE ----------
experience = []
try:
    exp_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'experience') or .//h2[contains(., 'Experience') or contains(., 'Esperienza')]]"
    )
    roles = exp_section.find_elements(By.XPATH, ".//li[.//h3 or .//span]")
    for role in roles:
        txt = role.text.strip()
        if txt:
            experience.append(txt)
except Exception:
    pass

# ---------- EDUCATION ----------
education = []
try:
    edu_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'education') or .//h2[contains(., 'Education') or contains(., 'Formazione')]]"
    )
    schools = edu_section.find_elements(By.XPATH, ".//li")
    for s in schools:
        txt = s.text.strip()
        if txt:
            education.append(txt)
except Exception:
    pass

# ---------- CLEANUP ----------
experience = clean_list(experience)
education = clean_list(education)

# ---------- RESULTS ----------
print("\n=== EXTRACTED FIELDS ===")
print("Name:", name)
print("Headline:", headline)
print("Location:", location)
print("About (preview):", safe_preview(about))
print(f"Experience entries: {len(experience)}")
print(f"Education entries: {len(education)}")

# ---------- SAVE OUTPUT ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"linkedin_profile_{ts}.csv"
out_json = f"linkedin_profile_{ts}.json"

fields = ["Name", "Headline", "Location", "About", "Experience", "Education"]
row = {
    "Name": name,
    "Headline": headline,
    "Location": location,
    "About": about,
    "Experience": " | ".join(experience),
    "Education": " | ".join(education)
}

# Save CSV
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerow(row)

# Save JSON
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(row, f, ensure_ascii=False, indent=2)

print(f"\n✅ Clean data saved to:\n  CSV → {os.path.abspath(out_csv)}\n  JSON → {os.path.abspath(out_json)}")

driver.quit()

Please login manually in the opened browser window. The script will continue automatically once you're logged in.
✅ Detected LinkedIn auth cookie 'li_at' — login successful.

=== EXTRACTED FIELDS ===
Name: Eliana Di Lodovico
Headline: PhD student in soil science
Location: Landau in der Pfalz, Rhineland-Palatinate, Germany
About (preview): None
Experience entries: 11
Education entries: 3

✅ Clean data saved to:
  CSV → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251008_143023.csv
  JSON → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251008_143023.json


In [16]:
import pandas as pd
from tabulate import tabulate

# --- Replace this with your actual CSV filename ---
csv_path = "linkedin_profile_20251008_143023.csv"  # e.g. "linkedin_profile_20251008_142533.csv"

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(csv_path)

# Show the first few rows (you’ll likely only have one)
print("\n=== Loaded LinkedIn Profile Data ===")
print(df.head())

# Print column names
print("\n=== Columns ===")
print(list(df.columns))

# Optionally, inspect individual fields
print("\nName:", df.loc[0, "Name"])
print("Headline:", df.loc[0, "Headline"])
print("Location:", df.loc[0, "Location"])
print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))


=== Loaded LinkedIn Profile Data ===
                 Name                     Headline  \
0  Eliana Di Lodovico  PhD student in soil science   

                                            Location  About  \
0  Landau in der Pfalz, Rhineland-Palatinate, Ger...    NaN   

                                          Experience  \
0  Researcher Assistant\nResearcher Assistant\nRP...   

                                           Education  
0  CodeOp\nCodeOp\nData Science\nData Science\nJa...  

=== Columns ===
['Name', 'Headline', 'Location', 'About', 'Experience', 'Education']

Name: Eliana Di Lodovico
Headline: PhD student in soil science
Location: Landau in der Pfalz, Rhineland-Palatinate, Germany
+--------------------+-----------------------------+----------------------------------------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [1]:
# ================================================================
# manual_login_and_extract_linkedin.py
# Run: python manual_login_and_extract_linkedin.py
# Requirements: pip install undetected-chromedriver selenium
# ================================================================

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, csv, json, os, sys
from datetime import datetime

# ---------- CONFIG ----------
profile_url = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"  # profile to scrape
driver_path = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"
wait_timeout = 300  # seconds to wait for manual login

# ---------- START BROWSER & MANUAL LOGIN ----------
options = uc.ChromeOptions()
options.add_argument("--start-maximized")  # open browser maximized

driver = uc.Chrome(driver_executable_path=driver_path, options=options)
driver.get("https://www.linkedin.com/login")
print("Please login manually in the opened browser window. The script will continue automatically once you're logged in.")

# Wait until LinkedIn session is detected
start_time = time.time()
while True:
    time.sleep(1)
    cookies = driver.get_cookies()
    if any(c.get("name") == "li_at" for c in cookies):
        print("✅ Detected LinkedIn auth cookie 'li_at' — login successful.")
        break
    if time.time() - start_time > wait_timeout:
        print(f"⚠️ No auth cookie detected after {wait_timeout} seconds.")
        ans = input("If you are already logged in, press Enter to continue anyway or type 'exit' to abort: ").strip().lower()
        if ans == "exit":
            driver.quit()
            sys.exit("Aborted by user.")
        else:
            break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(profile_url)
wait = WebDriverWait(driver, 15)

# ---------- UTILITIES ----------
def safe_click(xpath):
    """Safely click an element if it exists."""
    try:
        el = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].scrollIntoView(true);", el)
        time.sleep(0.3)
        el.click()
        time.sleep(0.5)
        return True
    except Exception:
        return False

def clean_list(lst):
    """Remove duplicates and empty strings while preserving order."""
    seen = set()
    cleaned = []
    for item in lst:
        if item and item.strip() and item not in seen:
            cleaned.append(item.strip())
            seen.add(item)
    return cleaned

def safe_preview(t, n=300):
    """Truncate long text for printing."""
    return (t[:n] + "...") if t and len(t) > n else t

def try_selectors(selectors):
    """Try multiple selectors until one returns non-empty text."""
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except Exception:
            continue
    return None

def get_section_text_by_header(header_texts):
    """Find section text by matching possible header titles."""
    for label in header_texts:
        try:
            xpath = f"//h2[contains(translate(normalize-space(.), '{label.upper()}', '{label.lower()}'), '{label.lower()}')]"
            header = driver.find_element(By.XPATH, xpath)
            section = header.find_element(By.XPATH, "./ancestor::section")
            text = section.text.strip()
            if "\n" in text:
                text = text.split("\n", 1)[1].strip()
            return text
        except Exception:
            continue
    return None

# ---------- EXPAND CONTENT ----------
# First, click any general "See more" / "Mostra altro" buttons
expand_xpaths = [
    "//button[contains(., 'See more')]",
    "//button[contains(., 'Mostra altro')]",
    "//button[contains(., 'Mostra di più')]",
    "//button[contains(., 'Vedi altro')]",
    "//button[contains(@aria-label, 'See more')]"
]
for xp in expand_xpaths:
    safe_click(xp)

# Scroll to load dynamic content
for frac in [0.25, 0.5, 0.75, 1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {frac});")
    time.sleep(0.7)

# ---------- EXPAND SPECIFIC SECTIONS ----------
# Some sections like Education and Experience are collapsed under "Show all..."
education_expand_xpaths = [
    "//button[contains(., 'Show all education')]",
    "//button[contains(., 'Mostra tutta la formazione')]",
    "//button[contains(., 'Mostra tutto')]",
    "//button[contains(., 'Vedi tutta la formazione')]",
    "//button[contains(@aria-label, 'Show all education')]"
]
for xp in education_expand_xpaths:
    safe_click(xp)

experience_expand_xpaths = [
    "//button[contains(., 'Show all experiences')]",
    "//button[contains(., 'Mostra tutte le esperienze')]",
    "//button[contains(., 'Vedi tutte le esperienze')]",
    "//button[contains(@aria-label, 'Show all experiences')]"
]
for xp in experience_expand_xpaths:
    safe_click(xp)

# Re-scroll after expanding sections
for frac in [0.25, 0.5, 0.75, 1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {frac});")
    time.sleep(1)

# ---------- EXTRACT FIELDS ----------
name = try_selectors([
    (By.CSS_SELECTOR, "h1.text-heading-xlarge"),
    (By.XPATH, "//main//h1")
])

headline = try_selectors([
    (By.CSS_SELECTOR, "div.text-body-medium.break-words"),
    (By.XPATH, "//main//div[contains(@class, 'text-body-medium')]")
])

location = try_selectors([
    (By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words"),
    (By.XPATH, "//main//span[contains(., ',') or contains(., 'Based')]")
])

about = get_section_text_by_header(["About", "Informazioni", "Profilo"])

# ---------- EXPERIENCE ----------
experience = []
try:
    exp_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'experience') or .//h2[contains(., 'Experience') or contains(., 'Esperienza')]]"
    )
    roles = exp_section.find_elements(By.XPATH, ".//li[.//h3 or .//span]")
    for role in roles:
        txt = role.text.strip()
        if txt:
            experience.append(txt)
except Exception:
    pass

# ---------- EDUCATION ----------
education = []
try:
    edu_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'education') or .//h2[contains(., 'Education') or contains(., 'Formazione')]]"
    )
    schools = edu_section.find_elements(By.XPATH, ".//li")
    for s in schools:
        txt = s.text.strip()
        if txt:
            education.append(txt)
except Exception:
    pass

# ---------- CLEANUP ----------
experience = clean_list(experience)
education = clean_list(education)

# ---------- RESULTS ----------
print("\n=== EXTRACTED FIELDS ===")
print("Name:", name)
print("Headline:", headline)
print("Location:", location)
print("About (preview):", safe_preview(about))
print(f"Experience entries: {len(experience)}")
print(f"Education entries: {len(education)}")

# ---------- SAVE OUTPUT ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"linkedin_profile_{ts}.csv"
out_json = f"linkedin_profile_{ts}.json"

fields = ["Name", "Headline", "Location", "About", "Experience", "Education"]
row = {
    "Name": name,
    "Headline": headline,
    "Location": location,
    "About": about,
    "Experience": " | ".join(experience),
    "Education": " | ".join(education)
}

# Save CSV
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerow(row)

# Save JSON
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(row, f, ensure_ascii=False, indent=2)

print(f"\n✅ Clean data saved to:\n  CSV → {os.path.abspath(out_csv)}\n  JSON → {os.path.abspath(out_json)}")

driver.quit()

Please login manually in the opened browser window. The script will continue automatically once you're logged in.
✅ Detected LinkedIn auth cookie 'li_at' — login successful.

=== EXTRACTED FIELDS ===
Name: Eliana Di Lodovico
Headline: PhD student in soil science
Location: Landau in der Pfalz, Rhineland-Palatinate, Germany
About (preview): None
Experience entries: 11
Education entries: 3

✅ Clean data saved to:
  CSV → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251021_185559.csv
  JSON → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251021_185559.json


In [3]:
import pandas as pd
from tabulate import tabulate

# --- Replace this with your actual CSV filename ---
csv_path = "linkedin_profile_20251021_185559.csv"  # e.g. "linkedin_profile_20251008_142533.csv"

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(csv_path)

# Show the first few rows (you’ll likely only have one)
print("\n=== Loaded LinkedIn Profile Data ===")
print(df.head())

# Print column names
print("\n=== Columns ===")
print(list(df.columns))

# Optionally, inspect individual fields
print("\nName:", df.loc[0, "Name"])
print("Headline:", df.loc[0, "Headline"])
print("Location:", df.loc[0, "Location"])
print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))


=== Loaded LinkedIn Profile Data ===
                 Name                     Headline  \
0  Eliana Di Lodovico  PhD student in soil science   

                                            Location  About  \
0  Landau in der Pfalz, Rhineland-Palatinate, Ger...    NaN   

                                          Experience  \
0  Researcher Assistant\nResearcher Assistant\nRP...   

                                           Education  
0  CodeOp\nCodeOp\nData Science\nData Science\nJa...  

=== Columns ===
['Name', 'Headline', 'Location', 'About', 'Experience', 'Education']

Name: Eliana Di Lodovico
Headline: PhD student in soil science
Location: Landau in der Pfalz, Rhineland-Palatinate, Germany
+--------------------+-----------------------------+----------------------------------------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
# ================================================================
# manual_login_and_extract_linkedin.py
# Run: python manual_login_and_extract_linkedin.py
# Requirements: pip install undetected-chromedriver selenium
# ================================================================

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, csv, json, os, sys, re
from datetime import datetime

# ---------- CONFIG ----------
profile_url = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"  # profile to scrape
driver_path = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"
wait_timeout = 300  # seconds to wait for manual login

# ---------- START BROWSER & MANUAL LOGIN ----------
options = uc.ChromeOptions()
options.add_argument("--start-maximized")  # open browser maximized

driver = uc.Chrome(driver_executable_path=driver_path, options=options)
driver.get("https://www.linkedin.com/login")
print("Please login manually in the opened browser window. The script will continue automatically once you're logged in.")

# Wait until LinkedIn session is detected
start_time = time.time()
while True:
    time.sleep(1)
    cookies = driver.get_cookies()
    if any(c.get("name") == "li_at" for c in cookies):
        print("✅ Detected LinkedIn auth cookie 'li_at' — login successful.")
        break
    if time.time() - start_time > wait_timeout:
        print(f"⚠️ No auth cookie detected after {wait_timeout} seconds.")
        ans = input("If you are already logged in, press Enter to continue anyway or type 'exit' to abort: ").strip().lower()
        if ans == "exit":
            driver.quit()
            sys.exit("Aborted by user.")
        else:
            break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(profile_url)
wait = WebDriverWait(driver, 15)

# ---------- UTILITIES ----------
def safe_click(xpath):
    """Safely click an element if it exists."""
    try:
        el = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].scrollIntoView(true);", el)
        time.sleep(0.3)
        el.click()
        time.sleep(0.5)
        return True
    except Exception:
        return False

def clean_list(lst):
    """Remove duplicates and empty strings while preserving order."""
    seen = set()
    cleaned = []
    for item in lst:
        if item and item.strip() and item not in seen:
            cleaned.append(item.strip())
            seen.add(item)
    return cleaned

def safe_preview(t, n=300):
    """Truncate long text for printing."""
    return (t[:n] + "...") if t and len(t) > n else t

def try_selectors(selectors):
    """Try multiple selectors until one returns non-empty text."""
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except Exception:
            continue
    return None

def get_section_text_by_header(header_texts):
    """Find section text by matching possible header titles."""
    for label in header_texts:
        try:
            xpath = f"//h2[contains(translate(normalize-space(.), '{label.upper()}', '{label.lower()}'), '{label.lower()}')]"
            header = driver.find_element(By.XPATH, xpath)
            section = header.find_element(By.XPATH, "./ancestor::section")
            text = section.text.strip()
            if "\n" in text:
                text = text.split("\n", 1)[1].strip()
            return text
        except Exception:
            continue
    return None

# ---------- EXPAND CONTENT ----------
# First, click any general "See more" / "Mostra altro" buttons
expand_xpaths = [
    "//button[contains(., 'See more')]",
    "//button[contains(., 'Mostra altro')]",
    "//button[contains(., 'Mostra di più')]",
    "//button[contains(., 'Vedi altro')]",
    "//button[contains(@aria-label, 'See more')]"
]
for xp in expand_xpaths:
    safe_click(xp)

# Scroll to load dynamic content
for frac in [0.25, 0.5, 0.75, 1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {frac});")
    time.sleep(0.7)

# ---------- EXPAND SPECIFIC SECTIONS ----------
# Some sections like Education and Experience are collapsed under "Show all..."
education_expand_xpaths = [
    "//button[contains(., 'Show all education')]",
    "//button[contains(., 'Mostra tutta la formazione')]",
    "//button[contains(., 'Mostra tutto')]",
    "//button[contains(., 'Vedi tutta la formazione')]",
    "//button[contains(@aria-label, 'Show all education')]"
]
for xp in education_expand_xpaths:
    safe_click(xp)

experience_expand_xpaths = [
    "//button[contains(., 'Show all experiences')]",
    "//button[contains(., 'Mostra tutte le esperienze')]",
    "//button[contains(., 'Vedi tutte le esperienze')]",
    "//button[contains(@aria-label, 'Show all experiences')]"
]
for xp in experience_expand_xpaths:
    safe_click(xp)

# Re-scroll after expanding sections
for frac in [0.25, 0.5, 0.75, 1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {frac});")
    time.sleep(1)

# ---------- EXTRACT FIELDS ----------
name = try_selectors([
    (By.CSS_SELECTOR, "h1.text-heading-xlarge"),
    (By.XPATH, "//main//h1")
])

headline = try_selectors([
    (By.CSS_SELECTOR, "div.text-body-medium.break-words"),
    (By.XPATH, "//main//div[contains(@class, 'text-body-medium')]")
])

location = try_selectors([
    (By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words"),
    (By.XPATH, "//main//span[contains(., ',') or contains(., 'Based')]")
])

about = get_section_text_by_header(["About", "Informazioni", "Profilo"])

# ---------- EXPERIENCE ----------
experience = []
try:
    exp_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'experience') or .//h2[contains(., 'Experience') or contains(., 'Esperienza')]]"
    )
    roles = exp_section.find_elements(By.XPATH, ".//li[.//h3 or .//span]")
    for role in roles:
        txt = role.text.strip()
        if txt:
            experience.append(txt)
except Exception:
    pass

# ---------- EDUCATION ----------
education = []
try:
    edu_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'education') or .//h2[contains(., 'Education') or contains(., 'Formazione')]]"
    )
    schools = edu_section.find_elements(By.XPATH, ".//li")
    for s in schools:
        txt = s.text.strip()
        if txt:
            education.append(txt)
except Exception:
    pass

# ---------- CLEANUP ----------
def strip_artifacts(text):
    """Remove trailing 'See more' or 'Mostra altro' artifacts from text."""
    text = re.sub(r"(See more|Mostra altro|Mostra di più|Mostra tutto|Vedi altro|Vedi di più)\s*$", "", text, flags=re.IGNORECASE)
    return text.strip()

experience = [strip_artifacts(t) for t in clean_list(experience)]
education = [strip_artifacts(t) for t in clean_list(education)]

# ---------- RESULTS ----------
print("\n=== EXTRACTED FIELDS ===")
print("Name:", name)
print("Headline:", headline)
print("Location:", location)
print("About (preview):", safe_preview(about))
print(f"Experience entries: {len(experience)}")
print(f"Education entries: {len(education)}")

# (Optional) Uncomment to visually debug the full education list:
# for e in education: print("-", e)

# ---------- SAVE OUTPUT ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"linkedin_profile_{ts}.csv"
out_json = f"linkedin_profile_{ts}.json"

fields = ["Name", "Headline", "Location", "About", "Experience", "Education"]
row = {
    "Name": name,
    "Headline": headline,
    "Location": location,
    "About": about,
    "Experience": " | ".join(experience),
    "Education": " | ".join(education)
}

# Save CSV
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerow(row)

# Save JSON
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(row, f, ensure_ascii=False, indent=2)

print(f"\n✅ Clean data saved to:\n  CSV → {os.path.abspath(out_csv)}\n  JSON → {os.path.abspath(out_json)}")

driver.quit()

Please login manually in the opened browser window. The script will continue automatically once you're logged in.
✅ Detected LinkedIn auth cookie 'li_at' — login successful.

=== EXTRACTED FIELDS ===
Name: Eliana Di Lodovico
Headline: PhD student in soil science
Location: Landau in der Pfalz, Rhineland-Palatinate, Germany
About (preview): None
Experience entries: 11
Education entries: 3

✅ Clean data saved to:
  CSV → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251022_123658.csv
  JSON → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251022_123658.json


In [5]:
import pandas as pd
from tabulate import tabulate

# --- Replace this with your actual CSV filename ---
csv_path = "linkedin_profile_20251022_123658.csv"  # e.g. "linkedin_profile_20251008_142533.csv"

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(csv_path)

# Show the first few rows (you’ll likely only have one)
print("\n=== Loaded LinkedIn Profile Data ===")
print(df.head())

# Print column names
print("\n=== Columns ===")
print(list(df.columns))

# Optionally, inspect individual fields
print("\nName:", df.loc[0, "Name"])
print("Headline:", df.loc[0, "Headline"])
print("Location:", df.loc[0, "Location"])
print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))


=== Loaded LinkedIn Profile Data ===
                 Name                     Headline  \
0  Eliana Di Lodovico  PhD student in soil science   

                                            Location  About  \
0  Landau in der Pfalz, Rhineland-Palatinate, Ger...    NaN   

                                          Experience  \
0  Researcher Assistant\nResearcher Assistant\nRP...   

                                           Education  
0  CodeOp\nCodeOp\nData Science\nData Science\nJa...  

=== Columns ===
['Name', 'Headline', 'Location', 'About', 'Experience', 'Education']

Name: Eliana Di Lodovico
Headline: PhD student in soil science
Location: Landau in der Pfalz, Rhineland-Palatinate, Germany
+--------------------+-----------------------------+----------------------------------------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
# ================================================================
# manual_login_and_extract_linkedin.py
# Run: python manual_login_and_extract_linkedin.py
# Requirements: pip install undetected-chromedriver selenium
# ================================================================

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import time, csv, json, os, sys, re
from datetime import datetime

# ---------- CONFIG ----------
PROFILE_URL = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"
CHROMEDRIVER_PATH = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"
WAIT_TIMEOUT = 300   # seconds to wait for manual login

# ---------- LAUNCH BROWSER ----------
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
driver = uc.Chrome(driver_executable_path=CHROMEDRIVER_PATH, options=options)
driver.get("https://www.linkedin.com/login")
print("Please log in manually in the browser window...")

# wait until login detected
start = time.time()
while True:
    time.sleep(1)
    if any(c.get("name") == "li_at" for c in driver.get_cookies()):
        print("✅ Login detected — proceeding.")
        break
    if time.time() - start > WAIT_TIMEOUT:
        input("⚠️ Timeout reached. Press Enter if you are already logged in.")
        break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(PROFILE_URL)
wait = WebDriverWait(driver, 15)

# ---------- UTILITIES ----------
def safe_click(xpath):
    """Click button safely if exists."""
    try:
        el = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].scrollIntoView(true);", el)
        time.sleep(0.3)
        el.click()
        time.sleep(0.7)
        return True
    except Exception:
        return False

def clean_list(seq):
    seen, out = set(), []
    for s in seq:
        s = s.strip()
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    return out

def strip_artifacts(txt):
    """Remove LinkedIn UI artifacts like 'See more' etc."""
    txt = re.sub(r"(See more|Mostra|Vedi).*", "", txt, flags=re.I)
    return txt.strip()

def try_selectors(selectors):
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except Exception:
            pass
    return None

# ---------- EXPAND ALL CONTENT ----------
expanders = [
    "//button[contains(.,'See more')]",
    "//button[contains(.,'Mostra')]",
    "//button[contains(@aria-label,'See more')]",
    "//button[contains(.,'Show all education')]",
    "//button[contains(.,'Show all experiences')]",
    "//button[contains(.,'Mostra tutta la formazione')]",
    "//button[contains(.,'Mostra tutte le esperienze')]",
]
for xp in expanders:
    safe_click(xp)

# Scroll through page to trigger dynamic loads
for frac in [0.25, 0.5, 0.75, 1]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{frac});")
    time.sleep(0.8)

# ---------- BASIC FIELDS ----------
name = try_selectors([(By.CSS_SELECTOR, "h1.text-heading-xlarge"), (By.XPATH, "//main//h1")])
headline = try_selectors([(By.CSS_SELECTOR, "div.text-body-medium.break-words")])
location = try_selectors([(By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words")])

# ---------- EXPERIENCE ----------
experience = []
try:
    exp_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'experience') or .//h2[contains(., 'Experience') or contains(., 'Esperienza')]]"
    )
    roles = exp_section.find_elements(By.XPATH, ".//li[.//h3 or .//span]")
    for role in roles:
        raw = role.text.strip()
        if not raw:
            continue
        # Split and clean lines
        lines = [l.strip() for l in raw.split("\n") if l.strip()]
        lines = [strip_artifacts(l) for l in lines]

        # Deduplicate similar lines (ignore punctuation, spaces, dots)
        unique, seen = [], set()
        for l in lines:
            norm = re.sub(r"[\s·\.]+", "", l.lower())
            if norm not in seen:
                seen.add(norm)
                unique.append(l)
        experience.append(" | ".join(unique))
except Exception as e:
    print("⚠️ Experience section error:", e)

# ---------- EDUCATION ----------
education = []
try:
    edu_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'education') or .//h2[contains(., 'Education') or contains(., 'Formazione')]]"
    )
    schools = edu_section.find_elements(By.XPATH, ".//li")
    for s in schools:
        raw = s.text.strip()
        if not raw:
            continue
        lines = [l.strip() for l in raw.split("\n") if l.strip()]
        lines = [strip_artifacts(l) for l in lines]
        unique, seen = [], set()
        for l in lines:
            norm = re.sub(r"[\s·\.]+", "", l.lower())
            if norm not in seen:
                seen.add(norm)
                unique.append(l)
        education.append(" | ".join(unique))
except Exception as e:
    print("⚠️ Education section error:", e)

experience = clean_list(experience)
education = clean_list(education)

# ---------- SAVE ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"linkedin_profile_{ts}.csv"
out_json = f"linkedin_profile_{ts}.json"

fields = ["Name", "Headline", "Location", "Experience", "Education"]
row = {
    "Name": name,
    "Headline": headline,
    "Location": location,
    "Experience": "\n".join(experience),
    "Education": "\n".join(education)
}

with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerow(row)

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(row, f, ensure_ascii=False, indent=2)

print(f"\n✅ Data saved to:\n  CSV → {os.path.abspath(out_csv)}\n  JSON → {os.path.abspath(out_json)}")

driver.quit()

Please log in manually in the browser window...
✅ Login detected — proceeding.

✅ Data saved to:
  CSV → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251022_125430.csv
  JSON → c:\Users\Utente\Desktop\CodeOp\Wdoit-Internship\linkedin_profile_20251022_125430.json


In [10]:
import pandas as pd
from tabulate import tabulate

df = pd.read_csv("linkedin_profile_20251022_125430.csv")
print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))

+--------------------+-----------------------------+----------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
import pandas as pd
from tabulate import tabulate

csv_path = "linkedin_profile_20251022_125430.csv"
df = pd.read_csv(csv_path)

# split multiline Experience / Education fields for readability
df_display = df.copy()
df_display["Experience"] = df["Experience"].str.split("\n")
df_display["Education"] = df["Education"].str.split("\n")

# tabulate wants strings, so join with real newlines
df_display["Experience"] = df_display["Experience"].apply(lambda x: "\n".join(x))
df_display["Education"] = df_display["Education"].apply(lambda x: "\n".join(x))

print("\n=== LINKEDIN PROFILE ===\n")
print(tabulate(df_display, headers="keys", tablefmt="fancy_grid", showindex=False))


=== LINKEDIN PROFILE ===

╒════════════════════╤═════════════════════════════╤════════════════════════════════════════════════════╤══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╤═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

In [12]:
# ================================================================
# manual_login_and_extract_linkedin.py
# Run: python manual_login_and_extract_linkedin.py
# Requirements: pip install undetected-chromedriver selenium pandas tabulate
# ================================================================

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import time, csv, json, os, sys, re
from datetime import datetime
import pandas as pd
from tabulate import tabulate

# ---------- CONFIG ----------
PROFILE_URL = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"
CHROMEDRIVER_PATH = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"
WAIT_TIMEOUT = 300   # seconds to wait for manual login

# ---------- LAUNCH BROWSER ----------
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
driver = uc.Chrome(driver_executable_path=CHROMEDRIVER_PATH, options=options)
driver.get("https://www.linkedin.com/login")
print("Please log in manually in the browser window...")

# wait until login detected
start = time.time()
while True:
    time.sleep(1)
    if any(c.get("name") == "li_at" for c in driver.get_cookies()):
        print("✅ Login detected — proceeding.")
        break
    if time.time() - start > WAIT_TIMEOUT:
        input("⚠️ Timeout reached. Press Enter if you are already logged in.")
        break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(PROFILE_URL)
wait = WebDriverWait(driver, 15)

# ---------- UTILITIES ----------
def safe_click(xpath):
    try:
        el = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].scrollIntoView(true);", el)
        time.sleep(0.3)
        el.click()
        time.sleep(0.7)
        return True
    except Exception:
        return False

def strip_artifacts(txt):
    txt = re.sub(r"(See more|Mostra|Vedi).*", "", txt, flags=re.I)
    return txt.strip()

def try_selectors(selectors):
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except Exception:
            pass
    return None

# ---------- EXPAND ALL CONTENT ----------
expanders = [
    "//button[contains(.,'See more')]",
    "//button[contains(.,'Mostra')]",
    "//button[contains(@aria-label,'See more')]",
    "//button[contains(.,'Show all education')]",
    "//button[contains(.,'Show all experiences')]",
    "//button[contains(.,'Mostra tutta la formazione')]",
    "//button[contains(.,'Mostra tutte le esperienze')]",
]
for xp in expanders:
    safe_click(xp)

# Scroll through page to trigger dynamic loads
for frac in [0.25, 0.5, 0.75, 1]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{frac});")
    time.sleep(0.8)

# ---------- BASIC FIELDS ----------
name = try_selectors([(By.CSS_SELECTOR, "h1.text-heading-xlarge"), (By.XPATH, "//main//h1")])
headline = try_selectors([(By.CSS_SELECTOR, "div.text-body-medium.break-words")])
location = try_selectors([(By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words")])
about = try_selectors([(By.XPATH, "//section[contains(@class,'pv-about-section')]//p")])

# ---------- EXPERIENCE (structured) ----------
experience = []
try:
    exp_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'experience') or .//h2[contains(., 'Experience') or contains(., 'Esperienza')]]"
    )

    roles = exp_section.find_elements(By.XPATH, ".//li[contains(@class, 'artdeco-list__item')]")
    for role in roles:
        raw_text = role.text.strip()
        if not raw_text:
            continue

        # Split by lines, clean, remove artifacts
        lines = [strip_artifacts(l.strip()) for l in raw_text.split("\n") if l.strip()]

        # Deduplicate normalized text
        seen_norm = set()
        clean_lines = []
        for l in lines:
            norm = re.sub(r"[\s·\.]+", "", l.lower())
            if norm not in seen_norm:
                seen_norm.add(norm)
                clean_lines.append(l)

        # Group by company → roles
        if len(clean_lines) > 3:
            company = clean_lines[1] if len(clean_lines) > 1 else "Unknown company"
            current_entry = {"company": company, "roles": []}
            current_role = None
            for l in clean_lines:
                if re.match(r"^[A-Z].+", l) and not any(x in l for x in ["·", "-", "to", "Present"]):
                    if current_role:
                        current_entry["roles"].append(current_role)
                    current_role = {"title": l, "details": ""}
                else:
                    if current_role:
                        current_role["details"] += (l + " ")
            if current_role:
                current_entry["roles"].append(current_role)
            experience.append(current_entry)
        else:
            experience.append({
                "company": clean_lines[1] if len(clean_lines) > 1 else "Unknown company",
                "roles": [{"title": clean_lines[0], "details": " ".join(clean_lines[2:])}],
            })
except Exception as e:
    print("⚠️ Error reading experience:", e)

# ---------- EDUCATION ----------
education = []
try:
    edu_section = driver.find_element(
        By.XPATH,
        "//section[contains(@id, 'education') or .//h2[contains(., 'Education') or contains(., 'Formazione')]]"
    )
    schools = edu_section.find_elements(By.XPATH, ".//li")
    for s in schools:
        raw_text = s.text.strip()
        if not raw_text:
            continue
        lines = [strip_artifacts(l.strip()) for l in raw_text.split("\n") if l.strip()]
        unique, seen_norm = [], set()
        for l in lines:
            norm = re.sub(r"[\s·\.]+", "", l.lower())
            if norm not in seen_norm:
                seen_norm.add(norm)
                unique.append(l)
        education.append(" | ".join(unique))
except Exception as e:
    print("⚠️ Error reading education:", e)

# ---------- FORMAT EXPERIENCE & EDUCATION ----------
def experience_to_text(experience):
    lines = []
    for company_block in experience:
        company = company_block.get("company", "Unknown Company")
        lines.append(f"🏢 {company}")
        for r in company_block["roles"]:
            title = r.get("title", "")
            details = r.get("details", "").strip()
            lines.append(f"   • {title} — {details}")
    return "\n".join(lines)

experience_text = experience_to_text(experience)
education_text = "\n".join(education)

# ---------- TERMINAL OUTPUT ----------
print("\n=== LINKEDIN PROFILE ===")
print(f"👤  Name: {name}")
print(f"💼  Headline: {headline}")
print(f"📍  Location: {location}\n")
if about:
    print(f"📝 About: {about}\n")
print("=== EXPERIENCE ===")
print(experience_text)
print("\n=== EDUCATION ===")
print(education_text)

# ---------- SAVE TO CSV / JSON ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"linkedin_profile_{ts}.csv"
out_json = f"linkedin_profile_{ts}.json"

row = {
    "Name": name,
    "Headline": headline,
    "Location": location,
    "About": about,
    "Experience": experience_text,
    "Education": education_text
}

# CSV
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    writer.writeheader()
    writer.writerow(row)

# JSON
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(row, f, ensure_ascii=False, indent=2)

print(f"\n✅ Data saved to:\n  CSV → {os.path.abspath(out_csv)}\n  JSON → {os.path.abspath(out_json)}")

driver.quit()

Please log in manually in the browser window...
✅ Login detected — proceeding.

=== LINKEDIN PROFILE ===
👤  Name: Eliana Di Lodovico
💼  Headline: PhD student in soil science
📍  Location: Landau in der Pfalz, Rhineland-Palatinate, Germany

=== EXPERIENCE ===
🏢 RPTU Kaiserslautern-Landau
   • Researcher Assistant — RPTU Kaiserslautern-Landau Feb 2022 - Present · 3 yrs 9 mos Feb 2022 to Present · 3 yrs 9 mos
🏢 Helmholtz Centre for Environmental Research
   • PHD Student — 
   • Helmholtz Centre for Environmental Research — Feb 2022 - Jan 2025 · 3 yrs Feb 2022 to Jan 2025 · 3 yrs
   • Leipzig — 
🏢 Full-time · 7 mos
   • Junior Environmental Consultant — Oct 2021 - Dec 2021 · 3 mos Oct 2021 to Dec 2021 · 3 mos
   • Castel San Giovanni, Emilia Romagna, Italia — 
   • Stage — Jun 2021 - Sep 2021 · 4 mos Jun 2021 to Sep 2021 · 4 mos
   • Consulenza ambientale — 
🏢 INRAE · Full-time
   • Studente tirocinante — INRAE · Full-time Jul 2019 - Sep 2019 · 3 mos Jul 2019 to Sep 2019 · 3 mos Bordeaux, 

In [ ]:
# ================================================================
# linkedin_scraper_heuristics.py
# Run: python linkedin_scraper_heuristics.py
# Requirements: pip install undetected-chromedriver selenium pandas tabulate
# ================================================================

import undetected_chromedriver as uc  # For automated Chrome browser that avoids detection
from selenium.webdriver.common.by import By  # For locating elements on page
import time, re, json, os  # Time for delays, re for regex, json/os for file handling
import pandas as pd  # For CSV and DataFrame handling
from tabulate import tabulate  # For printing tables nicely
from datetime import datetime  # For timestamped filenames

# ---------- CONFIG ----------
PROFILE_URL = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"  # LinkedIn profile URL
CHROMEDRIVER_PATH = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"  # Path to ChromeDriver
WAIT_TIMEOUT = 300   # Seconds to wait for manual login

# ---------- LAUNCH BROWSER ----------
options = uc.ChromeOptions()  # Initialize Chrome options
options.add_argument("--start-maximized")  # Start browser maximized
driver = uc.Chrome(driver_executable_path=CHROMEDRIVER_PATH, options=options)  # Launch browser
driver.get("https://www.linkedin.com/login")  # Open LinkedIn login page
print("Please log in manually in the browser window...")  # Prompt user

# Wait for login
start = time.time()  # Track starting time
while True:
    time.sleep(1)  # Wait 1 second between checks
    if any(c.get("name") == "li_at" for c in driver.get_cookies()):  # Check for LinkedIn login cookie
        print("✅ Login detected — proceeding.")  # Login successful
        break
    if time.time() - start > WAIT_TIMEOUT:  # Timeout reached
        input("⚠️ Timeout reached. Press Enter if already logged in.")  # Allow manual continuation
        break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(PROFILE_URL)  # Open target LinkedIn profile
time.sleep(2)  # Wait for page to load

# ---------- UTILITIES ----------
def safe_click(xpath):
    """Safely click an element if it exists."""
    try:
        el = driver.find_element(By.XPATH, xpath)  # Locate element
        driver.execute_script("arguments[0].scrollIntoView(true);", el)  # Scroll into view
        time.sleep(0.3)  # Short wait
        el.click()  # Click element
        time.sleep(0.7)  # Wait for action to complete
        return True  # Click successful
    except:
        return False  # Element not found / click failed

def clean_line(line):
    """Clean a line of text, remove 'See more' or similar."""
    line = re.sub(r"(See more|Mostra|Vedi).*", "", line, flags=re.I)  # Remove common LinkedIn expansion labels
    return line.strip()  # Remove leading/trailing spaces

# Expand buttons for extra content
expanders = [
    "//button[contains(.,'See more')]",
    "//button[contains(.,'Mostra')]",
    "//button[contains(@aria-label,'See more')]",
    "//button[contains(.,'Show all education')]",
    "//button[contains(.,'Show all experiences')]"
]
for xp in expanders:
    safe_click(xp)  # Try clicking each "see more" button

# Scroll page to load dynamic content
for f in [0.25,0.5,0.75,1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{f});")  # Scroll fractionally
    time.sleep(0.7)  # Wait for content to load

# ---------- BASIC INFO ----------
def try_selectors(selectors):
    """Try multiple CSS/XPath selectors and return first non-empty text."""
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except:
            pass
    return None

name = try_selectors([(By.CSS_SELECTOR,"h1.text-heading-xlarge"),(By.XPATH,"//main//h1")])  # Profile name
headline = try_selectors([(By.CSS_SELECTOR,"div.text-body-medium.break-words")])  # Headline
location = try_selectors([(By.CSS_SELECTOR,"span.text-body-small.inline.t-black--light.break-words")])  # Location
about = try_selectors([(By.XPATH,"//section[contains(@class,'pv-about-section')]//p")])  # About section

# ---------- EXPERIENCE ----------
# Finds Experience section on LinkedIn.
# Heuristic parsing.
# Limitations: for multi-role jobs in the same company, the heuristic may assign Role and Company incorrectly, because LinkedIn sometimes repeats the company name as the item title.

# Heuristic parsing means using simple rules or patterns to interpret text when there is no strict structure. Instead of relying on a formal schema, one makes educated guesses based on the content.
# I use it because LinkedIn profile sections like Experience or Education are not structured in consistent HTML fields. Each experience item is just a block of text.
# Use:
# - If a line contains a year → it’s Time/Duration.
# - If a line has a comma → likely Location.
# - First line that doesn’t match other rules → Role.
# - Next remaining line → Company.
# - Anything leftover → Description.

# What we see on LinkedIn is rendered for humans, however, the raw info are written, for example, as:
# <li class="artdeco-list__item lAivUhKtfDAdlfpDjPkzpXDITRwPnJUAgtnDmn fGslViEcTOTlZpKXphbfxSXaeSWMqrsAeGo">
              
          
# <!---->    <div class="mJVarBZbUrQBuvLhZeZTQSwIgeJwBFBbws
#         dabDCxDMqQxxAGhMIVSUGCGTrpPnkCMYZvxUc cCxRyVmQikrhQVfnYomBksTzkpriZOmqsghKqxE
        
        
        
#         " data-view-name="profile-component-entity">
#       <div>
        
#         <a data-field="experience_company_logo" class="optional-action-target-wrapper 
            
#             pvs-entity__image-container--outline-offset
#             display-flex" target="_self" href="https://www.linkedin.com/company/700268/">
              
#     <div class="ivm-image-view-model  pvs-entity__image ">
        
#     <div class="ivm-view-attr__img-wrapper
        
#         ">
# <!---->
# <!---->          <img width="48" src="https://media.licdn.com/dms/image/v2/C4E0BAQEKqHjp037FZw/company-logo_100_100/company-logo_100_100/0/1671636459738/technische_universitat_kaiserslautern_logo?e=1762992000&amp;v=beta&amp;t=Y-slWnv4e5vY_J9e7zp10t7cG-zdFcrcuV_BLVh1lGw" loading="lazy" height="48" alt="RPTU Kaiserslautern-Landau logo" id="ember879" class="ivm-view-attr__img--centered EntityPhoto-square-3   evi-image lazy-image ember-view">
#     </div>
  
#           </div>
  
#         </a>
  
#       </div>

#       <div class="display-flex flex-column align-self-center flex-grow-1">
#         <div class="display-flex flex-row justify-space-between">
          
#         <a data-field="experience_company_logo" class="optional-action-target-wrapper 
#               display-flex flex-column full-width" target="_self" href="https://www.linkedin.com/in/eliana-di-lodovico-570171192/add-edit/POSITION/?profileFormEntryPoint=PROFILE_SECTION&amp;entityUrn=urn%3Ali%3Afsd_profilePosition%3A%28ACoAAC067OQBxwr2iZkgnhLbCW4x1ji5EoBkZ8U%2C1908511197%29&amp;trackingId=umGUjd1aRyKmbl6zNc4%2F6w%3D%3D&amp;desktopBackground=MAIN_PROFILE">
#             <div class="display-flex flex-wrap align-items-center full-height">
                
#     <div class="display-flex ">
      
#       <div class="
#           display-flex full-width">
        
#           <div class="display-flex align-items-center
#               mr1 hoverable-link-text t-bold">
#             <span aria-hidden="true"><!---->Researcher Assistant<!----></span><span class="visually-hidden"><!---->Researcher Assistant<!----></span>
#           </div>
# <!---->      
#       </div>
      
#     </div>
  
# <!----><!----><!---->            </div>
#                 <span class="t-14 t-normal">
#                   <span aria-hidden="true"><!---->RPTU Kaiserslautern-Landau<!----></span><span class="visually-hidden"><!---->RPTU Kaiserslautern-Landau<!----></span>
#                 </span>
#               <span class="t-14 t-normal
#                   t-black--light">
#                 <span class="pvs-entity__caption-wrapper" aria-hidden="true"><!---->Feb 2022 - Present · 3 yrs 9 mos<!----></span><span class="visually-hidden"><!---->Feb 2022 to Present · 3 yrs 9 mos<!----></span>
#               </span>
# <!---->          </a>
  
#         </div>
# <!---->      </div>
# <!---->
# <!---->    </div>
  
  
#             </li>

experience = []
try:
    exp_section = driver.find_element(By.XPATH,
        "//section[contains(@id,'experience') or .//h2[contains(.,'Experience')]]")  # Experience section
    exp_items = exp_section.find_elements(By.XPATH,".//li")  # Each experience item
    for item in exp_items:
        lines = [clean_line(l) for l in item.text.split("\n") if clean_line(l)]  # Clean lines
        role, company, location_exp, time_text, duration, description = "", "", "", "", "", ""  # Initialize fields
        for l in lines:
            if re.search(r"\d{4}", l):  # Detect year/dates
                if "·" in l:
                    time_text,duration = [x.strip() for x in l.split("·",1)]  # Split date and duration
                else:
                    time_text = l
            elif ',' in l:  # If line contains comma, likely location
                location_exp = l
            elif not role:  # First remaining line is role
                role = l
            elif not company:  # Next line is company
                company = l
            else:  # Anything else is description
                description += l + " "
        experience.append({
            "Role": role,
            "Company": company,
            "Location": location_exp,
            "Time": time_text,
            "Duration": duration,
            "Description": description.strip()
        })
except Exception as e:
    print("⚠️ Experience extraction error:", e)  # Print error if extraction fails

# ---------- EDUCATION ----------
# Finds Education section and each entry
# Heuristic parsing: similar to experience.
# Limitations: complex degrees or multiple entries may not parse perfectly.

education = []
try:
    edu_section = driver.find_element(By.XPATH,
        "//section[contains(@id,'education') or .//h2[contains(.,'Education')]]")  # Education section
    edu_items = edu_section.find_elements(By.XPATH,".//li")  # Each education entry
    for item in edu_items:
        lines = [clean_line(l) for l in item.text.split("\n") if clean_line(l)]  # Clean lines
        school, degree, field, time_text = "", "", "", ""  # Initialize fields
        for l in lines:
            if re.search(r"\d{4}", l):  # If line has year, assign to time
                time_text = l
            elif any(x in l for x in ["Bachelor","Master","PhD","Laurea"]):  # Degree keywords
                degree = l
            elif not school:  # First remaining line is school
                school = l
            else:  # Anything else is field of study
                field = l
        education.append({
            "School": school,
            "Degree": degree,
            "Field": field,
            "Time": time_text
        })
except Exception as e:
    print("⚠️ Education extraction error:", e)  # Print error if extraction fails

# ---------- PRINT TABLES ----------
print("\n=== LINKEDIN PROFILE ===")
print(f"👤 Name: {name}")
print(f"💼 Headline: {headline}")
print(f"📍 Location: {location}")
if about:
    print(f"📝 About: {about}")

if experience:
    print("\n=== EXPERIENCE ===")
    exp_rows = [[e.get("Role",""),e.get("Company",""),e.get("Location",""),e.get("Time",""),
                 e.get("Duration",""),e.get("Description","")] for e in experience]
    exp_headers = ["Role","Company","Location","Time","Duration","Description"]
    print(tabulate(exp_rows, headers=exp_headers, tablefmt="fancy_grid"))  # Pretty table output

if education:
    print("\n=== EDUCATION ===")
    edu_rows = [[e.get("School",""),e.get("Degree",""),e.get("Field",""),e.get("Time","")] for e in education]
    edu_headers = ["School","Degree","Field","Time"]
    print(tabulate(edu_rows, headers=edu_headers, tablefmt="fancy_grid"))  # Pretty table output

# ---------- SAVE JSON & CSV ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")  # Timestamp for unique filenames
json_file = f"linkedin_profile_{ts}.json"
csv_exp = f"experience_{ts}.csv"
csv_edu = f"education_{ts}.csv"
csv_basic = f"basic_{ts}.csv"

with open(json_file,"w",encoding="utf-8") as f:  # Save full profile as JSON
    json.dump({"Name":name,"Headline":headline,"Location":location,"About":about,
               "Experience":experience,"Education":education}, f, ensure_ascii=False, indent=2)

pd.DataFrame(experience).to_csv(csv_exp,index=False)  # Save experience as CSV
pd.DataFrame(education).to_csv(csv_edu,index=False)  # Save education as CSV
pd.DataFrame([{"Name":name,"Headline":headline,"Location":location,"About":about}]).to_csv(csv_basic,index=False)  # Save basic info CSV

print(f"\n✅ JSON saved to {json_file}")
print(f"✅ Experience CSV saved to {csv_exp}")
print(f"✅ Education CSV saved to {csv_edu}")
print(f"✅ Basic info CSV saved to {csv_basic}")

driver.quit()  # Close browser

Please log in manually in the browser window...
✅ Login detected — proceeding.

=== LINKEDIN PROFILE ===
👤 Name: Eliana Di Lodovico
💼 Headline: PhD student in soil science
📍 Location: Landau in der Pfalz, Rhineland-Palatinate, Germany

=== EXPERIENCE ===
╒════════════════════════════════════════════╤════════════════════════════════════════════╤═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╤════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════